In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ==========================================
# 1. LOAD DATA
# ==========================================
processed_path = "../data/processed"
df = pd.read_csv(f"{processed_path}/master_features_final.csv")

# ==========================================
# 2. DEFINE FEATURES (X) AND TARGET (y)
# ==========================================
X = df.drop(columns=['account_id', 'churn_flag'])
y = df['churn_flag']

# ==========================================
# 3. TRAIN / TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================================
# 4. TRAIN RANDOM FOREST MODEL
# ==========================================
# n_estimators=100: Creates 100 decision trees and takes their majority vote.
# class_weight='balanced': Forces the model to penalize missed churners heavily!
# Note: No StandardScaler needed for tree-based models!
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# ==========================================
# 5. CROSS-VALIDATION (Reliability Check)
# ==========================================
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='roc_auc')
print(f"CV ROC-AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

# ==========================================
# 6. EVALUATE ON TEST SET
# ==========================================
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print("\n==========================================")
print("     RANDOM FOREST EVALUATION RESULTS     ")
print("==========================================")

print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_prob):.3f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    columns=['Predicted Retained (0)', 'Predicted Churned (1)'],
    index=['Actually Retained (0)', 'Actually Churned (1)']
))

CV ROC-AUC: 0.502 (+/- 0.066)

     RANDOM FOREST EVALUATION RESULTS     

ROC-AUC Score: 0.634

Classification Report:

              precision    recall  f1-score   support

           0       0.78      0.97      0.86        78
           1       0.00      0.00      0.00        22

    accuracy                           0.76       100
   macro avg       0.39      0.49      0.43       100
weighted avg       0.60      0.76      0.67       100


Confusion Matrix:

                       Predicted Retained (0)  Predicted Churned (1)
Actually Retained (0)                      76                      2
Actually Churned (1)                       22                      0


In [9]:
import pandas as pd

# ==========================================
# PROOF OF NO SIGNAL: CORRELATION ANALYSIS
# ==========================================
processed_path = "../data/processed"
df = pd.read_csv(f"{processed_path}/master_features_final.csv")

# HATA ÇÖZÜMÜ: account_id gibi metin (string) kolonlarını çöpe atıyoruz.
df_numeric = df.drop(columns=['account_id'])

# Sadece sayısal kolonlarla korelasyonları hesapla
correlations = df_numeric.corr()['churn_flag'].drop('churn_flag').abs().sort_values(ascending=False)

print("==========================================")
print("   TOP 10 FEATURE CORRELATIONS W/ CHURN   ")
print("==========================================")
print(correlations.head(10))

   TOP 10 FEATURE CORRELATIONS W/ CHURN   
industry_DevTools          0.117054
referral_source_event      0.096592
error_rate                 0.087243
referral_source_partner    0.083052
tenure_days                0.082401
industry_Cybersecurity     0.072421
total_usage_count          0.064186
total_usage_duration       0.063095
country_AU                 0.059968
referral_source_organic    0.058460
Name: churn_flag, dtype: float64
